### Scraping crawled URLs ;

#### Import libraries 

In [3]:
# libraries for data management 
import pandas as pd 
import json

# necessary libraries for scarping
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse 
from tqdm import tqdm
import time 

In [4]:
# function to scrape transcripts separately

def scrape_transcript(talk_url, headers):
    """
    Function to scrape the transcript from the talk url's transcript page at /transcript 
    """
    try:
        transcript_url = talk_url.rstrip('/') + "/transcript"
        response = requests.get(transcript_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        if not nextjs_data:
            return None 
        
        data = json.loads(nextjs_data.string)

        # we want to navigate to the transcript data once we have the foundation 
        transcript_data = data.get('props', {}).get('pageProps', {}).get('transcript')
        if not transcript_data:
            return None 

        # now join all transcript cues all together 
        cues = transcript_data.get('cues', [])
        full_text = " ".join([cue.get('text', '') for cue in cues if cue.get('text')])

        return full_text.strip()
    
    except Exception as e: 
        # we dont want to stop the whole script if a transcript fails 
        print(f" (Transcript Error: {str(e)[:30]}...)", end="")
        return None
    
    

In [12]:
# our main scraping function per ted talk url 

def scrape_single_ted_talk(url, headers):
    """ 
    Scrape all available keys (information on website) and the transcript from a single ted talk page url 
    """
    talk_details = {}
    transcript_data = {}

    try: 
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        nextjs_data = soup.find('script', id='__NEXT_DATA__')
        if not nextjs_data:
            return None, None 
        
        data = json.loads(nextjs_data.string)

        # navigate to the talk information 
        talk_data = data.get('props', {}).get('pageProps', {}).get('videoData', {})
        if not talk_data:
            talk_data = data.get('props', {}).get('pageProps', {})
        
        if not talk_data.get('id'):
            return None, None
        
        # lets extract talk details 
        talk_id = talk_data.get('id')
        talk_details = {
            'id': talk_id,
            'title': talk_data.get('title'),
            'speaker': talk_data.get('presenterDisplayName'),
            'description': talk_data.get('description'),
            'views': talk_data.get('viewedCount'),
            'duration': talk_data.get('duration'),                      # duration in seconds 
            'duration_minutes': talk_data.get('duration', 0) // 60,     # duration converted to minutes 
            'video_context': talk_data.get('videoContext'),
            'type': talk_data.get('type', {}).get('name'),
            'published_at': talk_data.get('publishedAt'),
            'url': talk_data.get('canonicalUrl'),
            'slug': talk_data.get('slug'),
            'tedcom_percentage': talk_data.get('tedcomPercentage'),
            'youtube_percentage': talk_data.get('youtubePercentage'),
            'podcasts_percentage': talk_data.get('podcastsPercentage')
        }

        # extracting topics 
        topics = talk_data.get('topics', {})
        if isinstance(topics, dict):
            topic_nodes = topics.get('nodes', [])
        else:
            topic_nodes = topics if isinstance(topics, list) else []
        
        topic_names = [t.get('name') for t in topic_nodes if isinstance(t, dict) and t.get('name')]
        talk_details['topics'] = ', '.join(topic_names)
        talk_details['number_of_topics'] = len(topic_names)

        # extracting transcripts 
        transcript_text = scrape_transcript(url, headers)
        if transcript_text: 
            transcript_data = {
                'talk_id': talk_id,                                     # foreign key to link 2 tables 
                'transcript': transcript_text
            }

        return talk_details, transcript_data 
    
    except Exception as e:
        print(f"Error scraping {url}: {str(e)}")
        return None, None
    


#### Load crawled dataset with existing urls 

In [13]:
# path of the first batch of urls: duration of 18+ minutes 
file_path = 'data/collected_csv/ted_talks18+.csv'

try: 
    df_input = pd.read_csv(file_path)
    print(f"Loaded {len(df_input)} URLs to scrape")

except FileNotFoundError:
    print(f"Error: file not found")
    df_input = pd.DataFrame()                      # creating empty dataframe if error 



Loaded 940 URLs to scrape


#### Start scraping

In [14]:
if not df_input.empty: 
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

    # two separate lists for our 2 dataframes (details & transcripts)
    all_talks_data = []
    all_transcripts_data = []

    # we're using tqdm library to visualize the scraping, to have a progress bar 
    for idx, row in tqdm(df_input.iterrows(), total=df_input.shape[0], desc="Scraping Talks..."):

        talk_data, transcript_data = scrape_single_ted_talk(row['url'], headers) 
        if talk_data:
            all_talks_data.append(talk_data)

        if transcript_data:
            all_transcripts_data.append(transcript_data)

        # respecting the server loads
        time.sleep(1)
    
    # save main talk details into new dataframe
    df_talks = pd.DataFrame(all_talks_data)

    # save transcript details into new dataframe 
    df_transcripts = pd.DataFrame(all_transcripts_data)


Scraping Talks...:   8%|▊         | 79/940 [07:31<1:16:49,  5.35s/it]

 (Transcript Error: ('Connection aborted.', Connec...)

Scraping Talks...: 100%|██████████| 940/940 [1:40:06<00:00,  6.39s/it]
